# Modelo enriquecido con datos cruzados

Enriquecemos el baseline del `ver_excel.ipynb` (R² = 0.44 ± 0.11 en CV) sumando dos fuentes que no usamos:

1. **`Chemical_Elements`** del Main (columna con los elementos químicos del depósito, 99.7% de cobertura — nunca la tocamos).
2. **`MED_ages_reordered_20200131.xlsx`**: 19431 muestras de edad isotópica con múltiples métodos de datación. Agregamos por `mindat_id` para obtener estadísticos (cantidad de muestras, media, std, cantidad de métodos).

No usamos Localities porque la mayoría de sus campos son códigos opacos (foreign keys sin tabla lookup) y sus campos de texto (`elements`, `mineral_association`) son redundantes con lo que ya tenemos en Main.

## 1. Cargar fuentes

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = "../data/external/Global-copper-deposit-dataset/"

df_main = pd.read_excel(DATA_DIR + "Global_Copper_Deposit_Main.xlsx")
df_ages = pd.read_excel(DATA_DIR + "MED_ages_reordered_20200131.xlsx")

print(f"Main:  {df_main.shape}")
print(f"Ages:  {df_ages.shape}")

Main:  (1487, 20)
Ages:  (19431, 33)


## 2. Base: depósitos con Cu reportado

In [2]:
df_base = df_main.dropna(subset=["Copper_grade(Cu; %)"]).copy()
df_base["Mindat_id"] = pd.to_numeric(df_base["Mindat_id"], errors="coerce").astype("Int64")
print(f"Base con Cu: {len(df_base)} depósitos")

Base con Cu: 989 depósitos


## 3. Enriquecer con MED_ages (edades isotópicas agregadas)

Agrupamos las 19431 muestras por `mindat_id` y calculamos estadísticos. Así cada depósito gana 6 features nuevas: cantidad de muestras datadas, media/std/min/max de la edad modelada, y cantidad de métodos distintos de datación.

In [3]:
df_ages["mindat_id"] = pd.to_numeric(df_ages["mindat_id"], errors="coerce").astype("Int64")
df_ages_clean = df_ages.dropna(subset=["mindat_id"])

age_agg = df_ages_clean.groupby("mindat_id").agg(
    n_age_samples=("modeled_age_ma", "size"),
    age_mean=("modeled_age_ma", "mean"),
    age_std=("modeled_age_ma", "std"),
    age_min=("modeled_age_ma", "min"),
    age_max=("modeled_age_ma", "max"),
    n_dating_methods=("dating_method", "nunique"),
).reset_index().rename(columns={"mindat_id": "Mindat_id"})

print(f"Depósitos en MED_ages: {len(age_agg)}")
print(f"\nMuestra de features agregadas:")
print(age_agg.head())

df_enriched = df_base.merge(age_agg, on="Mindat_id", how="left")
matched = df_enriched["n_age_samples"].notnull().sum()
print(f"\nDespués del merge: {df_enriched.shape}")
print(f"Depósitos con datos MED: {matched} ({100*matched/len(df_enriched):.1f}%)")

Depósitos en MED_ages: 6487

Muestra de features agregadas:
   Mindat_id  n_age_samples     age_mean    age_std  age_min  age_max  \
0         35              2  4566.315000   0.445477  4566.00  4566.63   
1         42              3   438.400000        NaN   438.40   438.40   
2         61              3  2862.333333  33.724373  2839.00  2901.00   
3         64              5     2.932000   0.848393     2.28     3.91   
4         65              1   630.000000        NaN   630.00   630.00   

   n_dating_methods  
0                 2  
1                 1  
2                 1  
3                 1  
4                 0  

Después del merge: (989, 26)
Depósitos con datos MED: 793 (80.2%)


## 4. Feature engineering

### 4.1 One-Hot de `Deposit_type`

In [4]:
df_model = pd.get_dummies(df_enriched, columns=["Deposit_type"], prefix="type")
print(f"Shape: {df_model.shape}")

Shape: (989, 30)


### 4.2 Features de minerales (top 50 de `Mineral_assemblage`)

In [5]:
from collections import Counter

all_minerals = []
for s in df_enriched["Mineral_assemblage"].dropna():
    all_minerals.extend([m.strip() for m in s.split(",")])

mineral_counts = Counter(all_minerals)
TOP_N_MIN = 50
top_minerals = [m for m, _ in mineral_counts.most_common(TOP_N_MIN)]

minerals_set = df_enriched["Mineral_assemblage"].apply(
    lambda s: set(m.strip() for m in s.split(",")) if pd.notna(s) else set()
)
for mineral in top_minerals:
    col = f"has_min_{mineral.lower().replace(' ', '_').replace('-', '_')}"
    df_model[col] = minerals_set.apply(lambda s: int(mineral in s))

print(f"Agregadas {len(top_minerals)} features de minerales")

Agregadas 50 features de minerales


### 4.3 Features de elementos químicos (top 30 de `Chemical_Elements`)

La columna viene como `-Al-Na-Si-O-...` — parseamos y creamos columnas binarias.

In [6]:
def parse_elements(s):
    if pd.isna(s):
        return set()
    return set(e.strip() for e in s.split("-") if e.strip())

elements_set = df_enriched["Chemical_Elements"].apply(parse_elements)

element_counts = Counter()
for s in elements_set:
    for e in s:
        element_counts[e] += 1

print(f"Elementos únicos: {len(element_counts)}")
print(f"\nTop 20 más frecuentes:")
for elem, count in element_counts.most_common(20):
    print(f"  {elem:4s}  {count:4d}  ({100*count/len(df_enriched):.1f}%)")

TOP_N_ELEM = 30
top_elements = [e for e, _ in element_counts.most_common(TOP_N_ELEM)]

for element in top_elements:
    df_model[f"elem_{element}"] = elements_set.apply(lambda s: int(element in s))

print(f"\nAgregadas {len(top_elements)} features de elementos")

Elementos únicos: 63

Top 20 más frecuentes:
  S      926  (93.6%)
  Cu     922  (93.2%)
  Fe     922  (93.2%)
  O      881  (89.1%)
  H      814  (82.3%)
  Si     800  (80.9%)
  Al     744  (75.2%)
  Ca     695  (70.3%)
  K      655  (66.2%)
  Zn     637  (64.4%)
  C      616  (62.3%)
  Mg     579  (58.5%)
  Pb     562  (56.8%)
  Ti     500  (50.6%)
  F      468  (47.3%)
  As     435  (44.0%)
  Mo     414  (41.9%)
  Au     412  (41.7%)
  Na     351  (35.5%)
  Sb     316  (32.0%)

Agregadas 30 features de elementos


## 5. Matriz de features final

In [7]:
feature_cols = (
    ["Latitude", "Longitude", "Tonnage(Mt)", "Max_age(Ma)", "Min_age(Ma)",
     "n_age_samples", "age_mean", "age_std", "age_min", "age_max", "n_dating_methods"]
    + [c for c in df_model.columns if c.startswith("type_")]
    + [c for c in df_model.columns if c.startswith("has_min_")]
    + [c for c in df_model.columns if c.startswith("elem_")]
)

X = df_model[feature_cols]
y = df_model["Copper_grade(Cu; %)"]
y_log = np.log1p(y)

print(f"Total features: {len(feature_cols)}")
print(f"  Numéricas base:      5")
print(f"  Edades MED agregadas: 6")
print(f"  One-hot tipo:        {sum(1 for c in feature_cols if c.startswith('type_'))}")
print(f"  Minerales:           {sum(1 for c in feature_cols if c.startswith('has_min_'))}")
print(f"  Elementos:           {sum(1 for c in feature_cols if c.startswith('elem_'))}")
print(f"\nShape X: {X.shape}")

Total features: 96
  Numéricas base:      5
  Edades MED agregadas: 6
  One-hot tipo:        5
  Minerales:           50
  Elementos:           30

Shape X: (989, 96)


## 6. Split train / val / test

In [8]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)
X_train, X_val, y_train_log, y_val_log = train_test_split(X_temp, y_temp_log, test_size=0.2, random_state=42)

y_train = np.expm1(y_train_log)
y_val = np.expm1(y_val_log)
y_test = np.expm1(y_test_log)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}  (INTOCABLE hasta la evaluación final)")

Train: (632, 96)
Val:   (159, 96)
Test:  (198, 96)  (INTOCABLE hasta la evaluación final)


## 7. Cross-validation con modelo baseline

Primero medimos con los mismos hiperparámetros del `ver_excel` para comparar **solo el efecto de las features nuevas**.

In [9]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

def rmse_original(y_true_log, y_pred_log):
    return np.sqrt(mean_squared_error(np.expm1(y_true_log), np.expm1(y_pred_log)))

def r2_original(y_true_log, y_pred_log):
    return r2_score(np.expm1(y_true_log), np.expm1(y_pred_log))

rmse_scorer = make_scorer(rmse_original, greater_is_better=False)
r2_scorer = make_scorer(r2_original, greater_is_better=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, random_state=42)

r2_scores = cross_val_score(cv_model, X_train, y_train_log, cv=kf, scoring=r2_scorer)
rmse_scores = -cross_val_score(cv_model, X_train, y_train_log, cv=kf, scoring=rmse_scorer)

print(f"R² por fold:    {r2_scores.round(3)}")
print(f"R² promedio:    {r2_scores.mean():.3f} ± {r2_scores.std():.3f}")
print(f"\nRMSE por fold:  {rmse_scores.round(3)}")
print(f"RMSE promedio:  {rmse_scores.mean():.3f} ± {rmse_scores.std():.3f}")
print(f"\n--- Baseline ver_excel.ipynb ---")
print(f"R² = 0.439 ± 0.106, RMSE = 1.007 ± 0.130")

R² por fold:    [0.632 0.275 0.449 0.427 0.385]
R² promedio:    0.434 ± 0.116

RMSE por fold:  [0.768 1.048 1.108 0.977 1.153]
RMSE promedio:  1.011 ± 0.135

--- Baseline ver_excel.ipynb ---
R² = 0.439 ± 0.106, RMSE = 1.007 ± 0.130


## 8. Búsqueda de hiperparámetros (RandomizedSearchCV)

In [10]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [200, 300, 500],
    "reg_alpha": [0, 0.1, 0.5, 1.0],
    "reg_lambda": [1, 3, 5, 10],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
}

search = RandomizedSearchCV(
    XGBRegressor(random_state=42), param_dist,
    n_iter=40, cv=kf, scoring=r2_scorer,
    random_state=42, n_jobs=-1, verbose=0,
)
search.fit(X_train, y_train_log)

print(f"Mejor R² CV:  {search.best_score_:.3f}")
print(f"\nMejores hiperparámetros:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

Mejor R² CV:  0.435

Mejores hiperparámetros:
  subsample: 0.8
  reg_lambda: 1
  reg_alpha: 0
  n_estimators: 300
  min_child_weight: 3
  max_depth: 4
  learning_rate: 0.03
  colsample_bytree: 0.7


## 9. Validación del modelo optimizado

In [11]:
best_params = search.best_params_
best_model_cv = XGBRegressor(**best_params, random_state=42)

r2_opt = cross_val_score(best_model_cv, X_train, y_train_log, cv=kf, scoring=r2_scorer)
rmse_opt = -cross_val_score(best_model_cv, X_train, y_train_log, cv=kf, scoring=rmse_scorer)

print(f"R² por fold:    {r2_opt.round(3)}")
print(f"R² promedio:    {r2_opt.mean():.3f} ± {r2_opt.std():.3f}")
print(f"\nRMSE por fold:  {rmse_opt.round(3)}")
print(f"RMSE promedio:  {rmse_opt.mean():.3f} ± {rmse_opt.std():.3f}")

R² por fold:    [0.584 0.297 0.423 0.469 0.401]
R² promedio:    0.435 ± 0.094

RMSE por fold:  [0.816 1.032 1.133 0.94  1.138]
RMSE promedio:  1.012 ± 0.122


## 10. Entrenamiento final + evaluación en test

In [12]:
X_train_full = pd.concat([X_train, X_val])
y_train_full_log = pd.concat([y_train_log, y_val_log])

best_model = XGBRegressor(**best_params, random_state=42)
best_model.fit(X_train_full, y_train_full_log)

y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"=== Test set (evaluación final) ===")
print(f"RMSE: {rmse:.3f}")
print(f"MAE:  {mae:.3f}")
print(f"R²:   {r2:.3f}")
print(f"\n--- ver_excel.ipynb (test) ---")
print(f"RMSE: 1.129, MAE: 0.574, R²: 0.368")

=== Test set (evaluación final) ===
RMSE: 1.127
MAE:  0.587
R²:   0.371

--- ver_excel.ipynb (test) ---
RMSE: 1.129, MAE: 0.574, R²: 0.368


## 11. Feature importance

In [13]:
importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("Top 25 features más importantes:")
for feat, imp in importances.head(25).items():
    print(f"  {feat:40s}  {imp:.4f}")

print(f"\n--- Distribución por grupo ---")
for prefix, label in [("type_", "Tipos"), ("has_min_", "Minerales"), ("elem_", "Elementos"),
                       ("", "Otras")]:
    if prefix:
        total = importances[importances.index.str.startswith(prefix)].sum()
    else:
        mask = ~importances.index.str.startswith(("type_", "has_min_", "elem_"))
        total = importances[mask].sum()
    print(f"  {label:12s}: {total:.3f}")

Top 25 features más importantes:
  type_Porphyry                             0.1842
  type_Magmatic sulfide                     0.0572
  type_Sediment-Hosted                      0.0482
  type_VMS                                  0.0372
  has_min_molybdenite                       0.0311
  elem_Ni                                   0.0259
  Tonnage(Mt)                               0.0232
  has_min_azurite                           0.0212
  has_min_chrysocolla                       0.0165
  has_min_malachite                         0.0150
  elem_K                                    0.0150
  elem_Co                                   0.0145
  elem_Mo                                   0.0137
  has_min_dolomite                          0.0124
  elem_Al                                   0.0115
  has_min_magnetite                         0.0107
  type_IOCG                                 0.0107
  elem_Te                                   0.0106
  elem_Zn                                   0.009